In [ ]:
import sys
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt

from src.features.build_features import OUTPUT_FIGURES_DIR, load_daily_data
from src.models.optimizer import (
    align_tick,
    calc_v_lp,
    get_amounts,
    get_liquidity,
    get_strategy_range,
    price_to_tick,
    run_regressions,
    tick_to_price,
)
from src.evaluation.backtest import export_backtest_results, run_backtest

df_daily = load_daily_data()
print("Columns:", df_daily.columns.tolist())
print("Recognized pools:", df_daily["pool_id"].unique().tolist())
display(df_daily.head(3))


In [ ]:
print("LP mathematics and strategy functions imported from src.models.optimizer.")


In [ ]:
df_ledger_out, df_decomp_out, df_metrics_out = run_backtest(df_daily)
export_backtest_results(df_ledger_out, df_decomp_out, df_metrics_out)

print("Backtest complete; ledger and evaluation tables saved under outputs/tables.")
display(df_metrics_out.head(10))


In [ ]:
reg_data, panel_results, df_quantiles, ols_model = run_regressions(
    df_daily, df_ledger_out
)

print(f"=== PANEL FIXED EFFECTS (R-squared: {ols_model.rsquared:.4f}) ===")
display(panel_results)
print("=== QUANTILE REGRESSION RESULTS ===")
display(df_quantiles.head(10))


In [ ]:
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Figure 4.3: return decomposition for pool P1.
fig, ax = plt.subplots(figsize=(10, 4.5), dpi=300)
p1_d = df_decomp_out[df_decomp_out['pool_id'] == 'P1']
x = np.arange(len(p1_d))
w = 0.2

ax.bar(x - 1.5*w, p1_d['Fees'], w, label='Fee Income ($)', color='#2ca02c')
ax.bar(x - 0.5*w, -p1_d['D_T'], w, label='-Inventory Loss ($)', color='#d62728')
ax.bar(x + 0.5*w, -p1_d['Gas'], w, label='-Gas Cost ($)', color='#ff7f0e')
ax.bar(x + 1.5*w, -p1_d['C_reb'], w, label='-Rebal Cost ($)', color='#9467bd')

ax.set_xticks(x)
ax.set_xticklabels(p1_d['strategy'], rotation=10)
ax.set_ylabel('USD')
ax.set_title('Figure 4.3: LP Position Return Decomposition (Pool P1: WETH/USDC 0.05%)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES_DIR / "figure_4_3_decomposition.png", dpi=300)
plt.show()

# Figure 4.5: cumulative equity curves.
fig, ax = plt.subplots(figsize=(10, 4.5), dpi=300)
for st in df_ledger_out[df_ledger_out['pool'] == 'P1']['strategy'].unique():
    sub = df_ledger_out[(df_ledger_out['pool'] == 'P1') & (df_ledger_out['strategy'] == st)]
    ax.plot(sub['timestamp'], sub['net_excess_return'] * 100, label=st, lw=1.8)

ax.axhline(0, color='black', linestyle=':', label='Benchmark HODL (ER = 0)')
ax.set_ylabel('Net Excess Return (%)')
ax.set_title('Figure 4.5: Cumulative Excess Return vs HODL (P1)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES_DIR / "figure_4_5_equity_curves.png", dpi=300)
plt.show()

print("All tables and figures have been saved under the repository outputs directory.")
